In [12]:
!nvidia-smi

Sun Nov 23 12:28:09 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.57                 Driver Version: 581.57         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060      WDDM  |   00000000:01:00.0 Off |                  N/A |
|  0%   30C    P8              5W /  145W |    6285MiB /   8151MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [1]:
%pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.3.221  Python-3.10.19 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
Setup complete  (28 CPUs, 31.8 GB RAM, 328.5/930.6 GB disk)


c:\miniconda3\envs\py\lib\site-packages\torch\cuda\__init__.py:215: UserWarning: 
NVIDIA GeForce RTX 5060 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90 compute_37.
If you want to use the NVIDIA GeForce RTX 5060 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [ ]:
# !! 極度危險：執行前請再次確認路徑 !!
# 刪除 /content/ 目錄下的 datasets 資料夾及其所有內容
!rm -rf /content

In [3]:
#下載資料集
import gdown
import os
import shutil

#下載training_image.
gdown.download("https://drive.google.com/uc?export=download&id=1bnbA-3XbhZfR0UbT44vgB8-kaQUQNeIo","/content/training_image.zip")
#下載training_label.
gdown.download("https://drive.google.com/uc?export=download&id=1N1YbDrEc23_-rbf9FDDW_VmeSL3IT4C_","/content/training_label.zip")
#下載訓練aortic_valve_colab.yaml
gdown.download("https://drive.google.com/uc?export=download&id=1D5z965VXyW4_jmilGm2zfNo787UA5lUB","/content/aortic_valve_colab.yaml")

Downloading...
From (original): https://drive.google.com/uc?export=download&id=1bnbA-3XbhZfR0UbT44vgB8-kaQUQNeIo
From (redirected): https://drive.google.com/uc?export=download&id=1bnbA-3XbhZfR0UbT44vgB8-kaQUQNeIo&confirm=t&uuid=62df358c-497b-4575-9aa4-06aaac2c354a
To: /content/training_image.zip
100%|██████████| 1.83G/1.83G [00:25<00:00, 71.7MB/s]
Downloading...
From: https://drive.google.com/uc?export=download&id=1N1YbDrEc23_-rbf9FDDW_VmeSL3IT4C_
To: /content/training_label.zip
100%|██████████| 659k/659k [00:00<00:00, 83.3MB/s]
Downloading...
From: https://drive.google.com/uc?export=download&id=1D5z965VXyW4_jmilGm2zfNo787UA5lUB
To: /content/aortic_valve_colab.yaml
100%|██████████| 91.0/91.0 [00:00<00:00, 427kB/s]


'/content/aortic_valve_colab.yaml'

In [3]:
import os
import shutil
import random

# --- 1. 尋找路徑與解壓縮 (保持您原本的邏輯) ---
def find_patient_root(root):
    for dirpath, dirnames, filenames in os.walk(root):
        if any(d.startswith("patient") for d in dirnames):
            return dirpath
    return root

if not os.path.isdir("./training_image") and os.path.exists("training_image.zip"):
    os.makedirs("./training_image", exist_ok=True)
    os.system("unzip -q training_image.zip -d ./training_image")

if not os.path.isdir("./training_label") and os.path.exists("training_label.zip"):
    os.makedirs("./training_label", exist_ok=True)
    os.system("unzip -q training_label.zip -d ./training_label")

IMG_ROOT = find_patient_root("./training_image")
LBL_ROOT = find_patient_root("./training_label")

print("IMG_ROOT =", IMG_ROOT)
print("LBL_ROOT =", LBL_ROOT)

# --- 2. 重建輸出資料夾 (確保乾淨) ---
# 如果想要重跑，建議先清空 datasets 資料夾
if os.path.exists("./datasets"):
    shutil.rmtree("./datasets")

os.makedirs("./datasets/train/images", exist_ok=True)
os.makedirs("./datasets/train/labels", exist_ok=True)
os.makedirs("./datasets/val/images", exist_ok=True)
os.makedirs("./datasets/val/labels", exist_ok=True)

# --- 3. 定義新的處理函數 (含 1:1 平衡與負樣本處理) ---
def process_dataset_balanced(start, end, split):
    print(f"\n正在處理 {split} 數據 (Patient {start:04d} - {end:04d})...")

    # --- A. 收集階段 ---
    all_positives = [] # 格式: (img_path, txt_path)
    all_negatives = [] # 格式: img_path (只有圖片)

    for i in range(start, end + 1):
        patient = f"patient{i:04d}"
        img_dir = os.path.join(IMG_ROOT, patient)
        lbl_dir = os.path.join(LBL_ROOT, patient)

        if not os.path.isdir(img_dir):
            continue

        # 遍歷該病人的所有圖片
        for img_fname in os.listdir(img_dir):
            if not img_fname.endswith(".png"):
                continue

            src_img_path = os.path.join(img_dir, img_fname)

            # 檢查標註檔是否存在
            base_name = os.path.splitext(img_fname)[0]
            txt_fname = base_name + ".txt"
            src_txt_path = os.path.join(lbl_dir, txt_fname)

            if os.path.exists(src_txt_path):
                all_positives.append((src_img_path, src_txt_path))
            else:
                all_negatives.append(src_img_path)

    print(f"  - 掃描結果: 正樣本 {len(all_positives)} 張, 潛在負樣本 {len(all_negatives)} 張")

    # --- B. 平衡與採樣階段 ---
    target_count = len(all_positives)

    # 隨機抽取負樣本以達到 1:1
    if len(all_negatives) > target_count:
        selected_negatives = random.sample(all_negatives, target_count)
        print(f"  - 執行平衡: 隨機抽取 {target_count} 張負樣本 (1:1)")
    else:
        selected_negatives = all_negatives
        print(f"  - 負樣本不足 ({len(all_negatives)} < {target_count})，使用全部負樣本")

    # --- C. 複製檔案與建立空標註 ---
    # 1. 處理正樣本
    for img_src, txt_src in all_positives:
        fname = os.path.basename(img_src)
        shutil.copy(img_src, f"./datasets/{split}/images/{fname}")
        shutil.copy(txt_src, f"./datasets/{split}/labels/{os.path.basename(txt_src)}")

    # 2. 處理負樣本 (建立空 txt)
    for img_src in selected_negatives:
        fname = os.path.basename(img_src)

        # 複製圖片
        shutil.copy(img_src, f"./datasets/{split}/images/{fname}")

        # 建立對應的空 txt 檔
        txt_name = os.path.splitext(fname)[0] + ".txt"
        dst_txt_path = f"./datasets/{split}/labels/{txt_name}"
        with open(dst_txt_path, 'w') as f:
            pass # 建立空檔案，代表無目標

    print(f"  -> {split} 處理完成！總共: {len(all_positives) + len(selected_negatives)} 張圖片")

# --- 4. 執行處理 ---
# Train: Patient 1-30 (會進行 1:1 平衡)
process_dataset_balanced(1, 30, "train")

# Val: Patient 31-50 (通常驗證集不需要強制 1:1，但這裡也會套用相同邏輯以保持分佈一致)
process_dataset_balanced(31, 50, "val")

IMG_ROOT = ./training_image\training_image
LBL_ROOT = ./training_label\training_label

正在處理 train 數據 (Patient 0001 - 0030)...
  - 掃描結果: 正樣本 1631 張, 潛在負樣本 8298 張
  - 執行平衡: 隨機抽取 1631 張負樣本 (1:1)
  -> train 處理完成！總共: 3262 張圖片

正在處理 val 數據 (Patient 0031 - 0050)...
  - 掃描結果: 正樣本 1156 張, 潛在負樣本 5778 張
  - 執行平衡: 隨機抽取 1156 張負樣本 (1:1)
  -> val 處理完成！總共: 2312 張圖片


In [6]:
!pip install opencv-python

In [ ]:
import cv2
import numpy as np
import os
import glob

# --- 1. 定義統一遮罩的歸一化座標 ---
# 使用您計算出來的靜態統一遮罩範圍
UNIFIED_MASK_NORM = {
    'x_min': 0.2140,
    'y_min': 0.3175,
    'x_max': 0.6005,
    'y_max': 0.7275
}

# --- 2. 定義統一遮罩函數 ---
def apply_unified_mask_and_save(image_path, norm_box):
    """
    對圖像應用固定的統一邊界框遮罩，將範圍外的部分變黑，並覆蓋原始檔案。
    """
    img = cv2.imread(image_path)
    if img is None:
        return

    H, W, _ = img.shape

    # 將歸一化座標轉換為像素座標
    x_min = int(norm_box['x_min'] * W)
    y_min = int(norm_box['y_min'] * H)
    x_max = int(norm_box['x_max'] * W)
    y_max = int(norm_box['y_max'] * H)

    # 確保座標在圖像範圍內
    x_min = max(0, x_min)
    y_min = max(0, y_min)
    x_max = min(W, x_max)
    y_max = min(H, y_max)

    # 創建一個全黑的遮罩
    mask = np.zeros_like(img, dtype=np.uint8)

    # 將統一邊界框區域設為白色 (255)
    mask[y_min:y_max, x_min:x_max] = 255

    # 應用遮罩
    masked_img = cv2.bitwise_and(img, mask)

    # 覆蓋原始圖像檔案
    cv2.imwrite(image_path, masked_img)

# --- 3. 執行遮罩處理於訓練和驗證集 ---
print("--- 開始對訓練和驗證數據執行統一靜態遮罩預處理 ---")

# 設定要處理的路徑
DATA_PATHS = ["./datasets/train/", "./datasets/val/"]

for base_path in DATA_PATHS:
    images_dir = os.path.join(base_path, "images")

    # 遍歷圖像資料夾中的所有 .png 圖像
    for img_path in glob.glob(os.path.join(images_dir, "*.png")):
        # 使用統一遮罩
        apply_unified_mask_and_save(img_path, UNIFIED_MASK_NORM)

print("--- 統一靜態遮罩預處理完成！訓練和驗證圖像已成功覆蓋。 ---")

--- 開始對訓練和驗證數據執行統一靜態遮罩預處理 ---
--- 統一靜態遮罩預處理完成！訓練和驗證圖像已成功覆蓋。 ---


In [4]:
!ls

'ls' ���O�����Υ~���R�O�B�i���檺�{���Χ妸�ɡC


In [5]:
print('訓練集圖片數量 : ',len(os.listdir("./datasets/train/images")))
print('訓練集標記數量 : ',len(os.listdir("./datasets/train/labels")))
print('驗證集圖片數量 : ',len(os.listdir("./datasets/val/images")))
print('驗證集標記數量 : ',len(os.listdir("./datasets/val/labels")))

訓練集圖片數量 :  3262
訓練集標記數量 :  3262
驗證集圖片數量 :  2312
驗證集標記數量 :  2312


In [1]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt') #初次訓練使用YOLO官方的預訓練模型，如要使用自己的模型訓練可以將'yolo12n.pt'替換掉
results = model.train(data="./aortic_valve_colab.yaml",
            epochs=50,
            batch=8,
            imgsz=640,
            patience=10,
            device=0 #使用GPU進行訓練
            )

New https://pypi.org/project/ultralytics/8.3.230 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.221  Python-3.10.19 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./aortic_valve_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train8, nbs=64, nms=False, opse

c:\miniconda3\envs\py\lib\site-packages\torch\cuda\__init__.py:215: UserWarning: 
NVIDIA GeForce RTX 5060 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90 compute_37.
If you want to use the NVIDIA GeForce RTX 5060 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       928  ultralytics.nn.modules.conv.Conv             [3, 32, 3, 2]                 
  1                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  2                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  3                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  4                  -1  1    103360  ultralytics.nn.modules.block.C3k2            [128, 256, 1, False, 0.25]    
  5                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  6                  -1  1    346112  ultralytics.nn.modules.block.C3k2            [256, 256, 1, True]           
  7                  -1  1   1180672  ultralytics

In [ ]:
!zip -r '/content/train.zip' '/content/runs/detect/train' #打包訓練模型和結果
from google.colab import files
files.download('/content/train.zip')

In [ ]:
# --- 1. 清理舊資料 (使用 shell 指令) ---
!rm -rf ./datasets
print("舊的 ./datasets 資料夾已清除。")


舊的 ./datasets 資料夾已清除。


'rm' ���O�����Υ~���R�O�B�i���檺�{���Χ妸�ɡC


NameError: name 'A' is not defined

In [ ]:
#預測階段
#下載資料集
import gdown
import os
!mkdir ./datasets
!mkdir ./datasets/test
gdown.download("https://drive.google.com/uc?export=download&id=15Wv0e5sQmrfArGDcAacwxBFujwvGIetu","/content/datasets/testing.zip")
!unzip '/content/datasets/testing' -d '/content/datasets/test/tmp'

Downloading...
From (original): https://drive.google.com/uc?export=download&id=15Wv0e5sQmrfArGDcAacwxBFujwvGIetu
From (redirected): https://drive.google.com/uc?export=download&id=15Wv0e5sQmrfArGDcAacwxBFujwvGIetu&confirm=t&uuid=77b8d785-0293-4a86-97ac-5f7882852e82
To: /content/datasets/testing.zip
100%|██████████| 1.83G/1.83G [00:21<00:00, 84.8MB/s]


串流輸出內容已截斷至最後 5000 行。
  inflating: /content/datasets/test/tmp/testing_image/patient0086/patient0086_0078.png  
  inflating: /content/datasets/test/tmp/testing_image/patient0086/patient0086_0079.png  
  inflating: /content/datasets/test/tmp/testing_image/patient0086/patient0086_0080.png  
  inflating: /content/datasets/test/tmp/testing_image/patient0086/patient0086_0081.png  
  inflating: /content/datasets/test/tmp/testing_image/patient0086/patient0086_0082.png  
  inflating: /content/datasets/test/tmp/testing_image/patient0086/patient0086_0083.png  
  inflating: /content/datasets/test/tmp/testing_image/patient0086/patient0086_0084.png  
  inflating: /content/datasets/test/tmp/testing_image/patient0086/patient0086_0085.png  
  inflating: /content/datasets/test/tmp/testing_image/patient0086/patient0086_0086.png  
  inflating: /content/datasets/test/tmp/testing_image/patient0086/patient0086_0087.png  
  inflating: /content/datasets/test/tmp/testing_image/patient0086/patient0086_0088.png  


In [ ]:
import os
import shutil

base_root = "/content/datasets/test/tmp"
dst_root1 = "/content/datasets/test/images1"
dst_root2 = "/content/datasets/test/images2"

os.makedirs(dst_root1, exist_ok=True)
os.makedirs(dst_root2, exist_ok=True)

# 自動找到第一個「直屬子資料夾含 patient*」的目錄
patient_root = base_root
for dirpath, dirnames, _ in os.walk(base_root):
    if any(d.lower().startswith("patient") for d in dirnames):
        patient_root = dirpath
        break

# 收集所有圖片路徑（只看直屬的 patient 資料夾）
all_files = []
for patient_folder in os.listdir(patient_root):
    patient_path = os.path.join(patient_root, patient_folder)
    if os.path.isdir(patient_path) and patient_folder.lower().startswith("patient"):
        for fname in os.listdir(patient_path):
            if fname.lower().endswith(".png"):
                all_files.append(os.path.join(patient_path, fname))

# 按名稱排序並對半移動
all_files.sort()
half = len(all_files) // 2

for f in all_files[:half]:
    shutil.move(f, os.path.join(dst_root1, os.path.basename(f)))

for f in all_files[half:]:
    shutil.move(f, os.path.join(dst_root2, os.path.basename(f)))

print(f"來源根目錄：{patient_root}")
print(f"完成移動！總共 {len(all_files)} 張，前半 {half} 張到 images1，後半 {len(all_files)-half} 張到 images2")

來源根目錄：/content/datasets/test/tmp/testing_image
完成移動！總共 16620 張，前半 8310 張到 images1，後半 8310 張到 images2


In [ ]:
import cv2
import numpy as np
import os
import glob

# --- 1. 遮罩座標 (保持不變) ---
UNIFIED_MASK_NORM = {
    'x_min': 0.2140,
    'y_min': 0.3175,
    'x_max': 0.6005,
    'y_max': 0.7275
}

# --- 2. 遮罩函數 (保持不變) ---
def apply_unified_mask_and_save(image_path, norm_box):
    img = cv2.imread(image_path)
    if img is None: return
    H, W, _ = img.shape
    x_min = int(norm_box['x_min'] * W)
    y_min = int(norm_box['y_min'] * H)
    x_max = int(norm_box['x_max'] * W)
    y_max = int(norm_box['y_max'] * H)
    x_min = max(0, x_min); y_min = max(0, y_min)
    x_max = min(W, x_max); y_max = min(H, y_max)
    mask = np.zeros_like(img, dtype=np.uint8)
    mask[y_min:y_max, x_min:x_max] = 255
    masked_img = cv2.bitwise_and(img, mask)
    cv2.imwrite(image_path, masked_img)

# --- 3. 設定路徑 (這裡改了！) ---
print("--- 開始對【測試集】執行統一靜態遮罩預處理 ---")

# ⚠️ 直接指向存放圖片的資料夾，不要只指到 test/
TEST_DIRECT_PATHS = [
    "./datasets/test/images1",
    "./datasets/test/images2"
]

for folder_path in TEST_DIRECT_PATHS:
    print(f"正在處理資料夾: {folder_path} ...")

    # 檢查資料夾是否存在
    if not os.path.exists(folder_path):
        print(f"❌ 找不到資料夾: {folder_path}")
        continue

    # 直接在該資料夾內找 png
    png_files = glob.glob(os.path.join(folder_path, "*.png"))

    if len(png_files) == 0:
        print("  ⚠️ 裡面沒有 PNG 圖片，請檢查路徑或副檔名。")
        continue

    for img_path in png_files:
        apply_unified_mask_and_save(img_path, UNIFIED_MASK_NORM)

    print(f"  ✅ 完成，處理了 {len(png_files)} 張圖片。")

print("--- 測試集遮罩處理全部完成！ ---")

--- 開始對【測試集】執行統一靜態遮罩預處理 ---
正在處理資料夾: ./datasets/test/images1 ...
  ✅ 完成，處理了 8310 張圖片。
正在處理資料夾: ./datasets/test/images2 ...
  ✅ 完成，處理了 8310 張圖片。
--- 測試集遮罩處理全部完成！ ---


In [ ]:
print('測試集圖片數量 : ',len(os.listdir("./datasets/test/images1"))+len(os.listdir("./datasets/test/images2")))

測試集圖片數量 :  16620


In [ ]:
#自行上傳權重檔請註解掉下方程式
gdown.download("https://drive.google.com/uc?export=download&id=1vkdlEbkD_3hFfZPkjHLF3m1BJte3UGmh","/content/best.pt")

Downloading...
From: https://drive.google.com/uc?export=download&id=1vkdlEbkD_3hFfZPkjHLF3m1BJte3UGmh
To: /content/best.pt
100%|██████████| 5.52M/5.52M [00:00<00:00, 29.7MB/s]


'/content/best.pt'

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/best.pt')
results = model.predict(source="./datasets/test/images1/",
              save=True,
              imgsz=640,
              device=0
              )

串流輸出內容已截斷至最後 5000 行。
image 3313/8310 /content/datasets/test/images1/patient0060_0359.png: 640x640 (no detections), 7.4ms
image 3314/8310 /content/datasets/test/images1/patient0060_0360.png: 640x640 (no detections), 8.2ms
image 3315/8310 /content/datasets/test/images1/patient0060_0361.png: 640x640 (no detections), 7.6ms
image 3316/8310 /content/datasets/test/images1/patient0060_0362.png: 640x640 (no detections), 10.9ms
image 3317/8310 /content/datasets/test/images1/patient0060_0363.png: 640x640 (no detections), 8.5ms
image 3318/8310 /content/datasets/test/images1/patient0060_0364.png: 640x640 (no detections), 7.4ms
image 3319/8310 /content/datasets/test/images1/patient0060_0365.png: 640x640 (no detections), 8.7ms
image 3320/8310 /content/datasets/test/images1/patient0060_0366.png: 640x640 (no detections), 7.4ms
image 3321/8310 /content/datasets/test/images1/patient0060_0367.png: 640x640 (no detections), 10.7ms
image 3322/8310 /content/datasets/test/images1/patient0060_0368.png: 640x640 

In [ ]:
print('預測類別 : ',results[260].boxes.cls[0].item())
print('預測信心分數 : ',results[260].boxes.conf[0].item())
print('預測框座標 : ',results[260].boxes.xyxy[0].tolist())

預測類別 :  0.0
預測信心分數 :  0.8704028129577637
預測框座標 :  [215.74917602539062, 244.3916473388672, 259.7255554199219, 313.038330078125]


In [ ]:
!mkdir ./predict_txt/
output_file = open('./predict_txt/images1.txt', 'w')
for i in range(len(results)):
    # 取得圖片檔名（不含副檔名）
    filename = results[i].path.split('/')[-1].split('.png')[0]

    # 取得預測框數量
    boxes = results[i].boxes
    box_num = len(boxes.cls.tolist())

    # 如果有預測框
    if box_num > 0:
        for j in range(box_num):
            # 提取資訊
            label = int(boxes.cls[j].item())  # 類別
            conf = boxes.conf[j].item()       # 信心度
            x1, y1, x2, y2 = boxes.xyxy[j].tolist()  # 邊界框座標

            # 建立一行資料
            line = f"{filename} {label} {conf:.4f} {int(x1)} {int(y1)} {int(x2)} {int(y2)}\n"
            output_file.write(line)

# 關閉輸出檔案
output_file.close()



In [ ]:
import torch ,gc

# 刪除大型變數
del boxes,all_files,results
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/best.pt')
results = model.predict(source="./datasets/test/images2/",
              save=True,
              imgsz=640,
              device=0
              )

串流輸出內容已截斷至最後 5000 行。
image 3313/8310 /content/datasets/test/images2/patient0086_0051.png: 640x640 (no detections), 7.6ms
image 3314/8310 /content/datasets/test/images2/patient0086_0052.png: 640x640 (no detections), 10.8ms
image 3315/8310 /content/datasets/test/images2/patient0086_0053.png: 640x640 (no detections), 9.6ms
image 3316/8310 /content/datasets/test/images2/patient0086_0054.png: 640x640 (no detections), 8.0ms
image 3317/8310 /content/datasets/test/images2/patient0086_0055.png: 640x640 (no detections), 8.0ms
image 3318/8310 /content/datasets/test/images2/patient0086_0056.png: 640x640 (no detections), 7.9ms
image 3319/8310 /content/datasets/test/images2/patient0086_0057.png: 640x640 (no detections), 9.8ms
image 3320/8310 /content/datasets/test/images2/patient0086_0058.png: 640x640 (no detections), 9.9ms
image 3321/8310 /content/datasets/test/images2/patient0086_0059.png: 640x640 (no detections), 7.8ms
image 3322/8310 /content/datasets/test/images2/patient0086_0060.png: 640x640 (

In [ ]:
output_file = open('./predict_txt/images2.txt', 'w')
for i in range(len(results)):
    # 取得圖片檔名（不含副檔名）
    filename = results[i].path.split('/')[-1].split('.png')[0]

    # 取得預測框數量
    boxes = results[i].boxes
    box_num = len(boxes.cls.tolist())

    # 如果有預測框
    if box_num > 0:
        for j in range(box_num):
            # 提取資訊
            label = int(boxes.cls[j].item())  # 類別
            conf = boxes.conf[j].item()       # 信心度
            x1, y1, x2, y2 = boxes.xyxy[j].tolist()  # 邊界框座標

            # 建立一行資料
            line = f"{filename} {label} {conf:.4f} {int(x1)} {int(y1)} {int(x2)} {int(y2)}\n"
            output_file.write(line)

# 關閉輸出檔案
output_file.close()


In [ ]:
file1 = "./predict_txt/images1.txt"
file2 = "./predict_txt/images2.txt"
output = "./predict_txt/merged.txt"

with open(output, "w", encoding="utf-8") as fout:
    for f in [file1, file2]:
        if os.path.exists(f):
            with open(f, "r", encoding="utf-8") as fin:
                fout.writelines(fin.readlines())

print(f"合併完成 -> {output}")


合併完成 -> ./predict_txt/merged.txt


In [ ]:
from google.colab import files
files.download('/content/predict_txt/merged.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>